![Banner](../images/09_07_banner_rs_texture.png)


# Texture Analysis

**Texture analysis** quantifies the spatial arrangement and variation of pixel intensities within an image — it captures pattern, roughness, regularity, and structure that spectral (color/brightness) values alone cannot.

Six notebooks take texture from a definition to a learned feature, all computed on a Landsat 9 scene over Las Vegas with `geolattice.rsense.texture`.

- **Texture Analysis.** This notebook. Why two patches with the same mean reflectance can still differ, and the families of methods that measure that difference.
- **Texture Analysis of Landsat Imagery: Statistical and Structural Methods.** First-order statistics, the gray-level co-occurrence matrix, run lengths, and structural primitives on the NIR band.
- **Texture Analysis of Landsat Imagery: Transform-Based (Spectral) Methods.** Fourier, Gabor, and wavelet texture, including the dominant orientation of the street grid and the mountain ridges.
- **Texture Analysis of Landsat Imagery: Local Pattern-Based Methods (LBP).** Local binary patterns and their variants, and where they hold up on the Las Vegas scene.
- **Texture Analysis of Landsat Imagery: Model-Based Methods.** Markov random fields, fractals, and autoregressive models fitted inside a local window.
- **Texture Analysis of Landsat Imagery: Deep-Learning-Based Methods.** Texture features from an ImageNet-pretrained ResNet-18, compared across layers and band choices.


## Overview

Two land patches can have identical mean reflectance (e.g., a smooth lake and a rippled sand dune under the same lighting) yet look completely different because of how their pixel values vary spatially. Texture analysis extracts that difference as quantitative features usable for classification, segmentation, or change detection.

In satellite image analysis it's especially valuable for:
- Separating classes with overlapping spectral signatures (e.g., urban vs. bare soil, different crop types, forest stand types)
- SAR imagery, where backscatter alone is noisy (speckle) but the *spatial pattern* of that speckle is informative (smooth water vs. rough forest canopy vs. textured urban fabric)
- Characterizing surface roughness, canopy structure, building density, ice/snow surface type


### Statistical Methods

The most widely used family. Based on the statistical distribution of gray-level (pixel intensity) values.

**a) First-order statistics (no spatial relationship)**
Computed directly from the image histogram within a moving window — mean, variance, standard deviation, skewness, kurtosis, entropy. Fast but ignores *where* pixels sit relative to each other, so two very differently-patterned textures can give identical first-order stats.

**b) Second-order statistics — Gray-Level Co-occurrence Matrix (GLCM / Haralick texture)**
The workhorse of satellite texture analysis. A GLCM tabulates how often pairs of gray levels occur at a specified distance and direction (0°, 45°, 90°, 135°) within a moving window. From that matrix, Haralick (1973) derived features such as:

| Feature | What it measures |
|---|---|
| Contrast | Local intensity variation (edges, roughness) |
| Correlation | Linear dependency between neighboring pixels |
| Energy / Angular Second Moment (ASM) | Uniformity — high for smooth, repetitive textures |
| Homogeneity (Inverse Difference Moment) | Closeness of the distribution to the diagonal |
| Entropy | Randomness/complexity of the texture |
| Dissimilarity | Average of absolute intensity differences |
| Variance | Spread of gray-level values |

GLCM texture is heavily used with SAR imagery (e.g., distinguishing forest, cropland, and urban areas by backscatter texture) and in `scikit-image`'s `graycomatrix`/`graycoprops`, ENVI, SNAP, and Orfeo Toolbox.

**c) Higher-order statistics**
- **Gray-Level Run Length Matrix (GLRLM):** measures runs of consecutive pixels with the same gray level in a given direction - good for directional/linear textures (e.g., furrowed fields, linear building rows).
- **Gray-Level Size Zone Matrix (GLSZM):** measures connected zones of same-intensity pixels regardless of direction — captures texture coarseness/fineness.


### Structural (Syntactic) Methods

Texture is treated as a set of primitive elements (texels) arranged according to placement rules - e.g., a regularly spaced row of houses or orchard trees. These methods detect repeating structural elements and their spatial arrangement rules, often via edge detection, morphological operations, or primitive-matching. Well suited to man-made, highly regular patterns (agricultural field rows, building grids) but fragile on natural, less-regular textures like vegetation canopy.


### Transform-Based (Spectral) Methods

Texture is analyzed in a transformed domain where repetitive patterns show up as concentrated energy at specific frequencies/scales.

- **Fourier Transform:** converts the image to the frequency domain; periodic textures (row crops, orchard grids) produce distinct peaks in the power spectrum. Sensitive to global periodicity but not localized texture changes.
- **Wavelet Transform:** decomposes the image into multiple scales and orientations simultaneously (multi-resolution analysis), capturing both coarse and fine texture detail — useful because real-world textures (forest canopy, urban fabric) exist at multiple scales at once.
- **Gabor Filters:** a bank of filters tuned to specific frequencies and orientations, modeling how texture varies directionally; widely used for crop row orientation, building facade detection, and as texture features ahead of classification (including as input channels to CNNs).


### Model-Based Methods

Texture is assumed to follow a generative statistical/geometric model, and the model's fitted parameters become the texture features.

- **Markov Random Fields (MRF):** model a pixel's value as statistically dependent on its neighbors; the fitted dependency parameters describe the texture. Commonly used in SAR/optical segmentation where spatial context regularizes noisy classification.
- **Fractal models:** estimate a fractal dimension describing surface roughness/self-similarity across scales — useful for natural terrain (coastlines, mountain texture, forest canopy roughness) where regular GLCM features struggle.
- **Autoregressive (AR) models:** express each pixel as a linear combination of its neighbors; the regression coefficients describe local texture.


### Local Pattern-Based Methods

- **Local Binary Patterns (LBP):** compares each pixel to its immediate neighbors, encoding the pattern as a binary code (brighter/darker than center), then builds a histogram of codes within a window. Computationally cheap, rotation-invariant variants exist, and it performs well for texture classification (e.g., urban building density, impervious surface detection) without needing GLCM's directional windows.


### Deep-Learning-Based Texture Features (modern approach)

Convolutional Neural Networks (CNNs) learn hierarchical texture representations automatically from data rather than hand-crafted statistics. Lower CNN layers tend to capture edge/texture-like features similar to Gabor/LBP responses, while deeper layers capture more abstract structure. Increasingly used for land-cover classification, crop-type mapping, and SAR texture classification where hand-crafted GLCM features previously dominated — though GLCM/LBP features are still commonly fused with CNN features or used standalone for interpretability and lower compute cost.


### Optical vs. SAR Texture - a key distinction

- **Optical imagery:** texture mainly reflects true surface structure (canopy gaps, building shadows, field furrows) since the sensor measures reflected sunlight.
- **SAR imagery:** backscatter texture is a mix of real surface roughness *and* speckle (a multiplicative noise artifact from coherent radar). This makes raw SAR pixel values noisy, but the *texture statistics* (GLCM contrast/homogeneity, local variance) of speckle patterns are themselves diagnostic — smooth water gives low-variance speckle, rough forest/urban areas give high-variance speckle — which is why GLCM-based texture is a standard companion to backscatter intensity in SAR land-cover classification (relevant to the RADARSAT/NISAR notebooks in this repo).


### Common tools for implementing texture analysis

- Python: `scikit-image` (`graycomatrix`/`graycoprops` for GLCM, `local_binary_pattern` for LBP), `mahotas`, `pyradiomics`
- GIS/remote sensing software: ENVI (texture toolbox), SNAP (GLCM operator for SAR), Orfeo Toolbox (`otbcli_HaralickTextureExtraction`), Google Earth Engine (`glcmTexture()`)
- R: `glcm` package

Want me to add a worked texture-analysis example (e.g., GLCM on one of the SAR scenes already in this repo) as a notebook cell or a short standalone demo?


**Installation.** This notebook needs installation option 1 (vector, raster, CRS and visualization): `pip install "geolattice[core]"`. See "Installation options" in the GeoLattice README.